In [3]:
import chromadb

In [4]:
#client=chromadb.Client()

In [5]:
#collection=client.create_collection(name="Ayurveda")

In [6]:
client=chromadb.PersistentClient(path="./ChromaVectorDB")

In [8]:
#collection=client.get_or_create_collection(name="Ayurveda")

In [13]:
collection=client.get_collection(name="Ayurveda")

In [14]:
#client.delete_collection(name="Ayurveda")

In [15]:
print(collection.count())

0


In [16]:
print("Collection Created:",collection.name)

Collection Created: Ayurveda


In [10]:
from pypdf import PdfReader

def read_pdf_pypdf(file_path):
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""  # Handle pages with no text
        return text.strip()
    except FileNotFoundError:
        return "Error: File not found."
    except Exception as e:
        return f"Error reading PDF: {e}"

# Example usage
if __name__ == "__main__":
    pdf_text = read_pdf_pypdf("Aryu2.pdf")
    #print(pdf_text)

In [11]:
from sentence_transformers import SentenceTransformer, util
def semantic_chunk(text, threshold=0.75):
   model = SentenceTransformer('all-MiniLM-L6-v2')
   sentences = text.split('. ')
   embeddings = model.encode(sentences, convert_to_tensor=True)
   chunks, current_chunk = [], [sentences[0]]
   for i in range(len(sentences)-1):
       sim = util.cos_sim(embeddings[i], embeddings[i+1])
       if sim < threshold:
           chunks.append(' '.join(current_chunk))
           current_chunk = [sentences[i+1]]
       else:
           current_chunk.append(sentences[i+1])
   chunks.append(' '.join(current_chunk))
   return chunks

C:\Users\LNMIIT\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
simple_chunks = semantic_chunk(pdf_text, threshold=0.75)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7231.56it/s]


In [13]:
print(len(simple_chunks))

152


In [14]:
# collection.add(
#     ids=[f"ch2_{i}" for i in range(len(simple_chunks))],
#     documents=simple_chunks
# )

In [15]:
BATCH_SIZE = 5000
ids=[f"ch1{i}" for i in range(len(simple_chunks))]
for i in range(0, len(simple_chunks), BATCH_SIZE):
    collection.add(
        documents=simple_chunks[i:i + BATCH_SIZE],
        ids=ids[i:i + BATCH_SIZE],
    )


In [16]:
print(collection.count())

152
